In [ ]:
# ===== 0. Cấu hình (chạy lần lượt 0 -> 6, runtime GPU L4 / A100 của Colab Pro)
BRANCH = "htstep_lora"
REPO_URL = "https://github.com/im-xiaoming/Omni-Retriever-Custom.git"
REPO = "/content/Omni-fix"
WORK = "/content/htstep_smoke"            # model, mp4, run (đĩa local của Colab)
BATCH_SIZES = [1, 2, 4, 8]
MAX_STEPS = 20
LIMIT_VIDEOS = (30, 5)                    # train (htstep_videos_4007.tar) / val, chỉ h264 / vp9
EVAL_ITEMS = 20
DRIVE_OUT = "/content/drive/MyDrive/Colab Notebooks/code KL/Omni/htstep_smoke"   # "" = không chép lên Drive

import os, re, json, glob, shutil, subprocess, statistics
WAVE_HOME = f"{WORK}/WAVE_HOME"
WAVE, BEATS = f"{WAVE_HOME}/WAVE-7B", f"{WAVE_HOME}/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt"
PT = f"{WORK}/adapters/omniretriever-7b"
VIDEO_ROOT, SMOKE = f"{WORK}/videos", f"{REPO}/data/htstep/smoke"
os.makedirs(WORK, exist_ok=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""       # repo đều public; token chỉ để tránh rate limit
except Exception:
    pass
ERRORS = []                               # lỗi gặp phải, in ở ô 6

def run(cmd, log, env=None):
    """Chạy cmd (bash), ghi log ra file, trả về (exit code, nội dung log)."""
    with open(log, "w") as f:
        rc = subprocess.run(["bash", "-lc", cmd], stdout=f, stderr=subprocess.STDOUT, env={**os.environ, **(env or {})}).returncode
    return rc, open(log, errors="replace").read()
print("WORK", WORK, "| BATCH_SIZES", BATCH_SIZES, "| MAX_STEPS", MAX_STEPS)

In [ ]:
# ===== 1. Code + thư viện + GPU
if not os.path.isdir(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
!cd {REPO} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q && git log -1 --format="commit %H %s"
!cd {REPO} && pip install -q -r training/requirements.txt && pip install -q decord av librosa soundfile einops "huggingface_hub[cli]"
import torch, transformers                 # không gọi torch.cuda ở kernel này: context CUDA sẽ giữ VRAM của các lượt train
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                   capture_output=True, text=True).stdout.strip()
assert q, "không có GPU: Runtime -> Change runtime type -> L4 / A100"
GPU, mib, cap = [x.strip() for x in q.splitlines()[0].split(",")]
VRAM_TOTAL = round(int(mib) / 1024, 2)
print("GPU", GPU, "| VRAM tổng", VRAM_TOTAL, "GB | compute", cap, "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| transformers", transformers.__version__)
if float(cap) < 8.0:                      # bf16 cần Ampere trở lên
    raise SystemExit("T4 không chạy được bf16, đổi sang L4 hoặc A100")

In [ ]:
# ===== 2. WAVE-7B + BEATs, adapter gốc của tác giả (kiểm sha256 với YunzeLiu/OmniRetriever-7B), vá RoPE
import hashlib
from huggingface_hub import HfApi
if not os.path.exists(f"{WAVE}/config.json"):
    !hf download nguyenminh04/omni-model --local-dir {WAVE_HOME} --quiet
if not os.path.exists(f"{PT}/adapter_model.safetensors"):
    !hf download nguyenminh04/omni-adapters --include "omniretriever-7b/*" --local-dir {WORK}/adapters --quiet
h = hashlib.sha256()
with open(f"{PT}/adapter_model.safetensors", "rb") as f:
    for b in iter(lambda: f.read(1 << 24), b""):
        h.update(b)
want = HfApi().get_paths_info("YunzeLiu/OmniRetriever-7B", ["adapter_model.safetensors"], expand=True)[0].lfs.sha256
print("CHECK adapter_is_author", "OK" if h.hexdigest() == want else "FAIL", h.hexdigest())
assert h.hexdigest() == want, "adapter khác bản của tác giả: dừng"
!cd {REPO} && python scripts/patch_wave_rope.py {WAVE}

In [ ]:
# ===== 3. mp4 nhỏ (HTTP range, chỉ đúng các member cần) + manifest train / val nhỏ
!cd {REPO} && python scripts/convert_htstep.py smoke --limit-videos {LIMIT_VIDEOS[0]} {LIMIT_VIDEOS[1]} --video-root {VIDEO_ROOT}
# manifest phải trùng bản đã commit (cùng luật, cùng codecs.jsonl)
!cd {REPO} && git diff --exit-code --stat data/htstep/smoke && echo "CHECK smoke_manifest_committed OK"
n_mp4 = len(glob.glob(f"{VIDEO_ROOT}/*.mp4"))
print("mp4:", n_mp4, "| train steps:", sum(1 for _ in open(f"{SMOKE}/smoke_train.jsonl")),
      "| val steps:", sum(1 for _ in open(f"{SMOKE}/smoke_val.jsonl")))

In [ ]:
# ===== 4. Train thử: mỗi BATCH_SIZE một tiến trình riêng (VRAM được trả lại), OOM thì ghi "OOM" rồi chạy tiếp
LORA_ONLY_LINE = re.compile(r"^\s+(\S+)\s+([\d,]+)\s*$")
RESULTS = {}
for bs in BATCH_SIZES:
    out = f"{WORK}/runs/bs{bs}"
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out)
    env = {"WAVE_PATH": WAVE, "BEATS_PATH": BEATS, "DATA_PATH": f"{SMOKE}/smoke_train.jsonl", "VIDEO_ROOT": VIDEO_ROOT,
           "LORA_CKPT": PT, "LORA_ONLY": "True", "LORA_INIT_ONLY": "True", "USE_TUPLE_INFONCE": "False",
           "GRADIENT_CHECKPOINTING": "True", "GRAD_ACCUM": "1", "NUM_GPUS": "1", "BATCH_SIZE": str(bs),
           "OUTPUT_DIR": out, "USE_DEEPSPEED": "False", "TRAIN_SEED": "42"}
    rc, log = run(f"cd {REPO} && bash training/train.sh --max_steps {MAX_STEPS} --save_steps {MAX_STEPS} "
                  f"--save_total_limit 1", f"{out}/train.log", env)
    # báo cáo tham số: khối "Trainable parameters by group" chỉ được có đúng một dòng "LoRA 6,881,280"
    block = log.split("Trainable parameters by group:")[-1].split("=" * 70)[0] if "by group" in log else ""
    groups = [m.groups() for m in map(LORA_ONLY_LINE.match, block.splitlines()) if m]
    params_ok = groups == [("LoRA", "6,881,280")]
    oom_skips = log.count("[OOM] Skipping")
    oom = "OutOfMemoryError" in log or "consecutive batches" in log or oom_skips > 0
    vram = json.load(open(f"{out}/vram.json")) if os.path.exists(f"{out}/vram.json") else {}
    steps = vram.get("step_seconds", [])[2:]                   # bỏ 2 bước đầu (khởi động)
    r = {"exit": rc, "params_report": groups, "params_ok": params_ok, "oom": oom, "oom_skipped_batches": oom_skips,
         "sample_errors": len(re.findall(r"\[(?:ERROR|TIMEOUT)\] Sample", log)),
         "after_load_gb": vram.get("after_load", {}).get("max_allocated_gb"),
         "peak_allocated_gb": max([x["max_allocated_gb"] for x in vram.get("log", [])], default=None),
         "peak_reserved_gb": max([x["max_reserved_gb"] for x in vram.get("log", [])], default=None),
         "sec_per_step": round(statistics.median(steps), 2) if steps else None, "steps_done": len(vram.get("step_seconds", [])),
         "losses": [x["loss"] for x in vram.get("log", []) if x.get("loss") is not None],
         "checkpoint": (glob.glob(f"{out}/checkpoint-*") or [None])[0]}
    if rc != 0 and not oom:
        tb = log[log.rfind("Traceback"):][-1500:] if "Traceback" in log else log[-1500:]
        ERRORS.append(f"bs{bs}: exit {rc}\n{tb}")
    if not params_ok:
        ERRORS.append(f"bs{bs}: báo cáo tham số khác 'LoRA 6,881,280': {groups}")
    if r["sample_errors"]:
        ERRORS.append(f"bs{bs}: {r['sample_errors']} mẫu lỗi (loader thay bằng mẫu ngẫu nhiên), xem {out}/train.log")
    RESULTS[bs] = r
    print(f"bs={bs}: exit {rc} | {'OOM' if oom else 'ok'} | params {groups} | sau load {r['after_load_gb']} GB | "
          f"đỉnh {r['peak_allocated_gb']} / reserved {r['peak_reserved_gb']} GB | {r['sec_per_step']} s/bước | "
          f"{r['steps_done']} bước | loss {r['losses'][:1]} -> {r['losses'][-1:]}", flush=True)

In [ ]:
# ===== 5. Eval thử: 20 bước val, batch 1, text + audio-video, checkpoint vừa train
ok_bs = [bs for bs in BATCH_SIZES if RESULTS.get(bs, {}).get("checkpoint") and not RESULTS[bs]["oom"]]
EVAL = None
if not ok_bs:
    ERRORS.append("eval: không có checkpoint nào (mọi batch đều lỗi / OOM)")
else:
    ck = RESULTS[max(ok_bs)]["checkpoint"]
    rc, log = run(f"cd {REPO} && python scripts/smoke_eval.py --wave {WAVE} --beats {BEATS} --adapter {ck} "
                  f"--manifest {SMOKE}/smoke_val.jsonl --video-root {VIDEO_ROOT} --n {EVAL_ITEMS} --out {WORK}/eval.json",
                  f"{WORK}/eval.log", {"BEATS_PATH": BEATS})
    if rc == 0:
        EVAL = json.load(open(f"{WORK}/eval.json"))
        print(json.dumps({k: EVAL[k] for k in ("adapter", "encoded", "failed", "after_load_gb", "eval_peak_allocated_gb",
                                                "eval_peak_reserved_gb", "ms_per_item")}, indent=1))
        if EVAL["failed"]:
            ERRORS.append(f"eval: {len(EVAL['failed'])} item không encode được: {EVAL['failed']}")
    else:
        ERRORS.append(f"eval: exit {rc}\n{log[log.rfind('Traceback'):][-1500:]}")

In [ ]:
# ===== 6. Bảng tóm tắt + vram_smoke_<GPU>.json
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
cell = lambda bs: ("OOM" if RESULTS[bs]["oom"] else f"{RESULTS[bs]['peak_allocated_gb']} GB") if bs in RESULTS else "-"
print(f"| GPU | VRAM tổng | sau khi load | " + " | ".join(f"đỉnh train bs{bs}" for bs in BATCH_SIZES)
      + " | giây / bước | đỉnh eval | lỗi |")
print("|---" * (5 + len(BATCH_SIZES)) + "|")
loads = [r["after_load_gb"] for r in RESULTS.values() if r["after_load_gb"] is not None]
print(f"| {GPU} | {VRAM_TOTAL} GB | {max(loads) if loads else '-'} GB | " + " | ".join(cell(bs) for bs in BATCH_SIZES)
      + " | " + ", ".join(f"bs{bs}: {r['sec_per_step']}" for bs, r in RESULTS.items())
      + f" | {EVAL['eval_peak_allocated_gb'] if EVAL else '-'} GB ({EVAL['ms_per_item'] if EVAL else '-'} ms/item) | {len(ERRORS)} |")
for e in ERRORS:
    print("LỖI:", e)
doc = {"gpu": GPU, "vram_total_gb": VRAM_TOTAL, "commit": commit, "torch": torch.__version__, "cuda": torch.version.cuda,
       "transformers": transformers.__version__, "max_steps": MAX_STEPS, "limit_videos": LIMIT_VIDEOS,
       "train": RESULTS, "eval": EVAL, "errors": ERRORS}
name = f"vram_smoke_{re.sub(r'[^A-Za-z0-9]+', '_', GPU)}.json"
json.dump(doc, open(f"{WORK}/{name}", "w"), indent=1)
print("->", f"{WORK}/{name}")
if DRIVE_OUT:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT, exist_ok=True)
    shutil.copy(f"{WORK}/{name}", DRIVE_OUT)
    for bs in RESULTS:
        shutil.copy(f"{WORK}/runs/bs{bs}/train.log", f"{DRIVE_OUT}/train_bs{bs}.log")
    for f in ("eval.json", "eval.log"):
        if os.path.exists(f"{WORK}/{f}"):
            shutil.copy(f"{WORK}/{f}", DRIVE_OUT)
    print("-> Drive:", DRIVE_OUT)